# Embedding divergence — the analysis reported in the paper

**CPU. About 30–40 minutes the first time (embedding); seconds afterwards, because per-event values are saved.**

This notebook produces the divergence numbers in the paper:

| Output | Section |
|---|---|
| Three arms: marker-matched revision vs marker-matched filler, each vs a random pool 4× the size of the revision arm, and all revisions vs the pool; clustered by trace and by (model, problem) | 2 |
| Robustness: the same comparison against a count-matched random draw | 3 |
| Decomposition: revision − filler = (revision − random) + (random − filler) | 3 |
| Revision vs filler with local repetition (`max_jaccard`) as a covariate, and with its interaction; clustered by trace, by (model, problem) and by problem alone (the strictest, used in the paper) | 4 |
| Repetition by arm, and its AUC for filler vs revision | 5 |

The random pool mirrors the revision arm's mix of sentence-boundary and mid-sentence positions. Marker matching is 1:1 on (model, first marker word), seed 42. Bootstrap: 2,000 cluster resamples, seed 8080. Results go to `hand_labeling_export/divergence_final.json`.

The arm builders are those of `embedding_divergence_regression_and_draw_stability.ipynb` (Part 2), embedded verbatim below. The comparison code is the code that produced the reported numbers.

**Prerequisites:** `labels/`, `rejects/`, `traces/`.

In [ ]:
# ---- Setup: Google Drive and the data folder ---------------------------------
import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/gdrive")

# Folder on Drive that holds traces/, labels/, activations/, rejects/ and
# hand_labeling_export/. Change it here if yours lives elsewhere.
DATA_ROOT = Path(os.environ.get("REVISION_DATA_ROOT",
                                "/content/gdrive/MyDrive/llm_planning_study"))
print("DATA_ROOT =", DATA_ROOT)

## 0 — Settings

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json, math, random, re, types

import numpy as np

GDRIVE_BASE = DATA_ROOT
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"
REJECT_DIR  = GDRIVE_BASE / "rejects"
LABELS_DIR  = GDRIVE_BASE / "labels"

N_BOOT      = 2000
SEED        = 8080
# Which correctness flag gates eligibility (correct and not truncated):
#   "saved"    the `correct` flag stored in the traces (written before the
#              ground-truth repair; this produced the reported numbers)
#   "repaired" `correct_new` from repaired_ground_truth.json (notebook 07: the
#              repaired answer key and the corrected answer extraction)
# "repaired" writes its own cache and result files and prints both side by side.
CORRECTNESS = "saved"
_sfx = "" if CORRECTNESS == "saved" else "_repaired_flag"
CACHE_PATH  = EXPORT_DIR / f"divergence_final_events{_sfx}.json"   # per-event values, written once
OUT         = EXPORT_DIR / f"divergence_final{_sfx}.json"
REUSE_CACHE = True      # False forces a full rebuild and re-embedding
R = {}

def hdr(t): print(f"\n{'='*78}\n{t}\n{'='*78}")

## 1 — Arm builders

In [ ]:
# ---- the arm builders: the draw-stability analysis, embedded verbatim -----------
# build_revision / build_pool / extract_window / max_jaccard / pack / eligible /
# load_trace and their constants (900/700-character budgets, 6-sentence caps,
# MIN_LEN = 2000, POOL_MULT = 4, all-MiniLM-L6-v2) are the functions of
# analyses/embedding_divergence_regression_and_draw_stability.ipynb, Part 2.
# They are executed into their own namespace `ds`, so nothing below can shadow
# them (several helper names below are reused with different bodies).
DS_SOURCE = r'''
# =============================================================================
# DIVERGENCE — DRAW STABILITY, A MIXTURE-MATCHED BASELINE, AND A LARGE POOL
#
# THIS SUPERSEDES divergence_matched_random_arm.py, WHOSE VERDICT WAS WRONG.
# That script printed "CONFIRMED, AND LARGER" while its own section 1 reported
# that matching had FAILED (4/4 structural covariates still imbalanced). The
# verdict was never gated on whether the matching worked. It is here.
#
# =============================================================================
# THREE PROBLEMS THE LAST RUN EXPOSED
#
# P1. THE ESTIMATE IS UNSTABLE ACROSS RANDOM DRAWS  *** the serious one ***
#     Two runs of the same code with the same seed gave
#         is_revision +0.0147 [+0.0060, +0.0235]
#         is_revision +0.0093 [+0.0005, +0.0175]
#     because the second script also drew a matched arm, consuming rng state, so
#     the uniform arm's POSITIONS differed. A 37% move, and the second interval
#     barely clears zero.
#
#     The bootstrap resamples the DRAWN events. It never resamples the DRAW. So
#     the random arm's position-sampling variance appears in NO confidence
#     interval reported so far -- a second way the CIs have been too narrow,
#     unrelated to clustering.
#
#     FIX, two parts:
#       (a) stop matching the random arm's COUNT to the revision arm. There was
#           never a reason to -- unbalanced n is fine in regression -- and a
#           revision-sized baseline is a small sample of positions whose noise
#           lands straight in the estimate. A POOL several times larger gives
#           the FULL-POOL estimate a smaller standard error.
#           (Precisely: it shrinks the SE of the full-pool fit. It does NOT
#           shrink the spread across revision-sized subsets, which is a
#           property of n=2649, not of the pool. Those are different quantities
#           and conflating them would be wrong.)
#       (b) measure the draw variance instead of assuming it away: resample
#           revision-sized subsets from the pool many times, refit each, and
#           report the DISTRIBUTION of the estimate across draws. That is what
#           characterises how much a count-matched design wobbles. If the
#           distribution straddles zero, the effect is not established however
#           tight any single run's bootstrap CI looked.
#
#     A CALIBRATION NOTE WORTH KNOWING. Under an i.i.d. simulation at this n and
#     noise level, two independent draws differ by about 0.002 typically. The
#     two real runs differed by 0.0054 -- roughly 2-3x more. That excess is what
#     you expect when observations are CLUSTERED (341 traces, ~15.5 events each),
#     because the effective sample size is far below 2649. So draw instability
#     and clustering compound, and the across-draw spread below is measured on
#     the real clustered data rather than assumed from an i.i.d. formula.
#
# P2. THE MATCHED ARM OVERSHOT
#     Revision events are a MIXTURE: ~79% sit at a sentence boundary, ~21%
#     mid-sentence (markers like "...the answer, actually wait"). Drawing the
#     baseline only at boundaries gave frag_before = 0.000 against revision's
#     0.208, replacing a +0.755 imbalance with a -0.208 one, and it broke
#     n_before, which had been balanced.
#     FIX: mirror the mixture. Draw at a boundary with the revision arm's own
#     boundary probability, and mid-sentence otherwise.
#
# P3. ATTRITION AND A REPETITION FLIP
#     The matched arm filled only 2026 of 2649 (76%) -- the boundary constraint
#     inside [1000, len-800] failed often, so it was drawn from a biased subset
#     of traces. And its max_jaccard came out 0.4473 against revision's 0.4274,
#     i.e. MORE repetitive than revision, where the uniform arm was LESS. Since
#     divergence falls with repetition, a more-repetitive baseline has lower
#     divergence and INFLATES the gap -- so "matched is larger" may be that
#     flip rather than the fragment fix.
#     FIX: report attrition explicitly, and report the estimate both raw and
#     with repetition controlled, so the flip cannot masquerade as an effect.
#
# COST: CPU, ~25-35 minutes (the pool is POOL_MULT x larger than before).
# =============================================================================



from pathlib import Path
from collections import Counter, defaultdict
import json, random, re

import numpy as np
from scipy.stats import mannwhitneyu
from sentence_transformers import SentenceTransformer

GDRIVE_BASE = DATA_ROOT
LABELS_DIR  = GDRIVE_BASE / "labels"
TRACES_DIR  = GDRIVE_BASE / "traces"
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"
OUT_PATH    = EXPORT_DIR / "divergence_draw_stability.json"

CHAR_BUDGET_BEFORE, CHAR_BUDGET_AFTER = 900, 700
RAW_BUFFER_BEFORE,  RAW_BUFFER_AFTER  = 2500, 1200
N_BEFORE = N_AFTER = 6
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
SEED, MIN_LEN, NQ = 8080, 2000, 4
N_BOOT = 1500
POOL_MULT = 4          # random pool = POOL_MULT x revision count
N_DRAWS = 40           # how many revision-sized subsets to refit on
P_FLOOR = 1.0 / N_BOOT

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B",  "DeepSeek-R1-Distill-Qwen-14B"]
def clean_bpe(t): return t.replace("Ġ", " ").replace("Ċ", "\n")
def split_sentences(t): return [p for p in re.split(r'(?<=[.!?])\s+', t.strip()) if p.strip()]

def take_by_budget(sents, budget, from_end):
    if not sents: return []
    ordered = list(reversed(sents)) if from_end else sents
    out, total = [], 0
    for s in ordered:
        out.append(s); total += len(s) + 1
        if total >= budget: break
    return list(reversed(out)) if from_end else out

def extract_window(fr, cs, ce):
    rs = max(0, cs - RAW_BUFFER_BEFORE); re_ = min(len(fr), ce + RAW_BUFFER_AFTER)
    w = clean_bpe(fr[rs:re_]); ls, le = cs - rs, ce - rs
    if not (0 <= ls <= le <= len(w)): return None, None
    before = take_by_budget(split_sentences(w[:ls]), CHAR_BUDGET_BEFORE, True)
    after  = take_by_budget(split_sentences(w[le:]), CHAR_BUDGET_AFTER, False)
    if after and not re.search(r'[.!?]\s*$', after[-1]): after = after[:-1]
    before, after = before[-N_BEFORE:], after[:N_AFTER]
    if not before or not after: return None, None
    return before, after

def eligible(tr):
    return tr is not None and tr.get("correct", False) and not tr.get("truncated", False)

def load_trace(model, pid, cache):
    k = (model, pid)
    if k not in cache:
        p = TRACES_DIR / model / f"{pid}.json"
        cache[k] = json.load(open(p)) if p.exists() else None
    return cache[k]

_TOK = re.compile(r"[a-z0-9]+")
def toks(s): return set(_TOK.findall(s.lower()))

def max_jaccard(before, after):
    bs = [toks(s) for s in before]; as_ = [toks(s) for s in after]
    best = 0.0
    for b in bs:
        if not b: continue
        for a in as_:
            if not a: continue
            i = len(b & a)
            if i:
                j = i / len(b | a)
                if j > best: best = j
    return float(best)

def fmt_p(p): return f"<{P_FLOOR:.4f}" if p < P_FLOOR else f"{p:.4g}"

def sentence_starts(fr):
    return [m.end() for m in re.finditer(r'(?<=[.!?])\s+', fr)]

def pack(arm, model, pid, before, after, cs, fr, tr):
    return {"arm": arm, "model": model, "problem_id": pid,
            "trace_id": f"{model}/{pid}", "before": before, "after": after,
            "n_before": len(before), "n_after": len(after),
            "frag_before": 0 if re.search(r'[.!?]["\')\]]?\s*$', before[-1]) else 1,
            "frag_after": 0 if re.match(r'^["\'(\[]?[A-Z0-9]', after[0].strip()) else 1,
            "rel_pos": cs / max(1, len(fr)),
            "difficulty": 1 if tr.get("difficulty") == "hard" else 0}


def build_revision(cache):
    rev, mlens = [], []
    for model in MODELS:
        mdir = LABELS_DIR / model
        if not mdir.exists(): continue
        for f in mdir.glob("*.json"):
            obj = json.load(open(f)); pid = obj.get("problem_id", f.stem)
            revs = [r for r in obj.get("revisions", []) or [] if r.get("verified", True)]
            if not revs: continue
            tr = load_trace(model, pid, cache)
            if not eligible(tr): continue
            fr = tr.get("full_response", "")
            if len(fr) < MIN_LEN: continue
            for r in revs:
                cs, ce = r.get("char_start"), r.get("char_end")
                if cs is None or ce is None or not (0 <= cs < len(fr)): continue
                b, a = extract_window(fr, cs, ce)
                if b is None: continue
                mlens.append(ce - cs)
                rev.append(pack("revision", model, pid, b, a, cs, fr, tr))
    return rev, np.array(mlens)


def build_pool(cache, rng, per_model_target, mlens, p_boundary):
    """A LARGE random pool mirroring the revision arm's boundary/mid-sentence
    MIXTURE, with a marker-sized span excised in both cases. Returns the pool
    and the attrition record."""
    pool, attr = [], Counter()
    for model, n_want in per_model_target.items():
        pids = [pid for (m, pid), tr in cache.items() if m == model and eligible(tr)]
        if not pids: continue
        made = att = 0
        cap = n_want * 60
        while made < n_want and att < cap:
            att += 1
            pid = pids[rng.randrange(len(pids))]
            tr = cache[(model, pid)]; fr = tr.get("full_response", "")
            if len(fr) < MIN_LEN:
                attr["too short"] += 1; continue
            lo, hi = 1000, len(fr) - 800
            if hi <= lo:
                attr["no usable range"] += 1; continue
            # MIXTURE: boundary with the revision arm's own probability
            if rng.random() < p_boundary:
                bounds = [x for x in sentence_starts(fr) if lo <= x <= hi]
                if not bounds:
                    attr["no boundary in range"] += 1; continue
                cs = bounds[rng.randrange(len(bounds))]
                kind = "boundary"
            else:
                cs = rng.randrange(lo, hi)
                kind = "midsentence"
            mlen = int(mlens[rng.randrange(len(mlens))])
            ce = min(cs + mlen, len(fr) - 1)
            b, a = extract_window(fr, cs, ce)
            if b is None:
                attr["window failed"] += 1; continue
            ev = pack("random", model, pid, b, a, cs, fr, tr)
            ev["draw_kind"] = kind
            pool.append(ev); made += 1
        if made < n_want:
            attr[f"{model} short by {n_want-made}"] += 1
    return pool, attr
def _ls(y, X):
    b, *_ = np.linalg.lstsq(X, y, rcond=None); return b

def fit_boot(y, X, groups=None, n_boot=N_BOOT, seed=0):
    beta = _ls(y, X); rng = np.random.RandomState(seed)
    n = len(y); draws = np.empty((n_boot, X.shape[1]))
    if groups is None:
        for b in range(n_boot):
            i = rng.randint(0, n, n); draws[b] = _ls(y[i], X[i])
    else:
        gi = defaultdict(list)
        for i, g in enumerate(groups): gi[g].append(i)
        arrs = [np.array(v) for v in gi.values()]; G = len(arrs)
        for b in range(n_boot):
            i = np.concatenate([arrs[j] for j in rng.randint(0, G, G)])
            draws[b] = _ls(y[i], X[i])
    return beta, draws

def ci_p(col):
    lo, hi = np.percentile(col, [2.5, 97.5])
    p = 2 * min((col <= 0).mean(), (col >= 0).mean())
    return float(lo), float(hi), float(min(1.0, p))


def design(ev, control_repetition=True):
    y = np.array([e["divergence"] for e in ev], float)
    r = np.array([1.0 if e["arm"] == "revision" else 0.0 for e in ev])
    cols = [np.ones_like(y), r]
    cm = None
    if control_repetition:
        c = np.array([e["max_jaccard"] for e in ev], float)
        cm = float(c.mean()); cc = c - cm
        cols += [cc, r * cc]
    return y, np.column_stack(cols), [e["trace_id"] for e in ev], cm


def main():
    cache, results = {}, {}
    print(f"Loading {EMBED_MODEL_NAME} ...")
    emb = SentenceTransformer(EMBED_MODEL_NAME)

    rev, mlens = build_revision(cache)
    p_boundary = 1.0 - float(np.mean([e["frag_before"] for e in rev]))
    print(f"  revision {len(rev)} events, {len({e['trace_id'] for e in rev})} traces")
    print(f"  marker length median {np.median(mlens):.0f} chars")
    print(f"  revision boundary rate (1 - frag_before) = {p_boundary:.3f}")
    print(f"  -> the pool will mirror this mixture, not force all boundaries")

    per_model = {m: c * POOL_MULT for m, c in Counter(e["model"] for e in rev).items()}
    print(f"\n  building a pool {POOL_MULT}x the revision count "
          f"({sum(per_model.values())} target) ...")
    pool, attr = build_pool(cache, random.Random(SEED), per_model, mlens, p_boundary)
    print(f"  pool: {len(pool)} events  ({len(pool)/sum(per_model.values()):.0%} of target)")
    if attr:
        print("  attrition:")
        for k, v in attr.most_common(6): print(f"    {v:>6}  {k}")
    results["pool"] = {"n": len(pool), "target": sum(per_model.values()),
                       "p_boundary": p_boundary, "attrition": dict(attr)}

    print(f"\n  embedding {len(rev)+len(pool)} windows (this is the slow part) ...")
    for e in rev + pool:
        B = emb.encode(e["before"], normalize_embeddings=True)
        A = emb.encode(e["after"],  normalize_embeddings=True)
        cb, ca = B.mean(axis=0), A.mean(axis=0)
        e["divergence"] = 1.0 - float(np.dot(cb, ca) /
                                      (np.linalg.norm(cb) * np.linalg.norm(ca)))
        e["max_jaccard"] = max_jaccard(e["before"], e["after"])

    # ---------------- P2: did the mixture match? ---------------------------
    print(f"\n{'#'*74}\n# 1 — DID THE MIXTURE MATCH? (P2)\n{'#'*74}")
    print(f"  {'covariate':<14}{'revision':>11}{'pool':>11}{'gap':>10}{'p':>11}")
    bal, struct_bad = {}, 0
    for k in ("n_before", "n_after", "frag_before", "frag_after", "rel_pos",
              "difficulty", "max_jaccard"):
        a = np.array([e[k] for e in rev], float); b = np.array([e[k] for e in pool], float)
        _, p = mannwhitneyu(a, b, alternative="two-sided")
        bal[k] = {"rev": float(a.mean()), "pool": float(b.mean()),
                  "gap": float(a.mean()-b.mean()), "p": float(p)}
        if k in ("n_before", "n_after", "frag_before", "frag_after") and p < 0.05:
            struct_bad += 1
        print(f"  {k:<14}{a.mean():>11.4f}{b.mean():>11.4f}"
              f"{a.mean()-b.mean():>+10.4f}{p:>10.3g}{'*' if p<0.05 else ' '}")
    results["balance"] = bal
    print(f"\n  structural covariates imbalanced: {struct_bad}/4")
    matched_ok = struct_bad <= 1
    print(f"  -> {'acceptably matched' if matched_ok else 'STILL IMBALANCED — treat the raw estimate as confounded'}")

    # ---------------- P1: draw stability ----------------------------------
    print(f"\n{'#'*74}\n# 2 — DRAW STABILITY (P1, the serious one)\n{'#'*74}")
    print(f"  Refitting on {N_DRAWS} independent revision-sized subsets of the pool.")
    print(f"  Each subset is drawn per-model to match the revision arm's model mix.")
    by_model = defaultdict(list)
    for e in pool: by_model[e["model"]].append(e)
    need = Counter(e["model"] for e in rev)
    rng = np.random.RandomState(0)
    ests = {"raw": [], "rep": []}
    for d in range(N_DRAWS):
        sub = []
        for m, n in need.items():
            avail = by_model[m]
            if not avail: continue
            idx = rng.choice(len(avail), size=min(n, len(avail)), replace=False)
            sub += [avail[i] for i in idx]
        for tag, ctrl in (("raw", False), ("rep", True)):
            y, X, tid, _ = design(rev + sub, control_repetition=ctrl)
            ests[tag].append(float(_ls(y, X)[1]))
    draw = {}
    for tag, lab in (("raw", "no repetition control"),
                     ("rep", "repetition controlled")):
        a = np.array(ests[tag])
        lo, hi = np.percentile(a, [2.5, 97.5])
        frac_pos = float((a > 0).mean())
        draw[tag] = {"mean": float(a.mean()), "sd": float(a.std()),
                     "range": [float(a.min()), float(a.max())],
                     "pct2_5": float(lo), "pct97_5": float(hi),
                     "frac_positive": frac_pos}
        print(f"\n  --- is_revision across draws, {lab} ---")
        print(f"    mean {a.mean():+.4f}   sd {a.std():.4f}")
        print(f"    range [{a.min():+.4f}, {a.max():+.4f}]")
        print(f"    2.5-97.5 pct [{lo:+.4f}, {hi:+.4f}]")
        print(f"    positive in {frac_pos:.0%} of draws")
    results["draw_stability"] = draw

    # ---------------- the pooled estimate on the FULL pool -----------------
    print(f"\n{'#'*74}\n# 3 — ESTIMATE ON THE FULL POOL (draw noise minimised)\n{'#'*74}")
    full = {}
    for tag, ctrl in (("raw", False), ("rep", True)):
        y, X, tid, cm = design(rev + pool, control_repetition=ctrl)
        b, dd = fit_boot(y, X, groups=tid)
        lo1, hi1, p1 = ci_p(dd[:, 1])
        entry = {"is_revision": {"beta": float(b[1]), "ci95": [lo1, hi1], "p": p1}}
        lab = "no repetition control" if tag == "raw" else "repetition controlled"
        print(f"\n  --- {lab}  (n_rev={len(rev)}, n_pool={len(pool)}) ---")
        print(f"    is_revision {b[1]:>+9.4f}  [{lo1:+.4f}, {hi1:+.4f}]  p={fmt_p(p1)}")
        if ctrl:
            lo3, hi3, p3 = ci_p(dd[:, 3])
            entry["interaction"] = {"beta": float(b[3]), "ci95": [lo3, hi3], "p": p3}
            print(f"    interaction {b[3]:>+9.4f}  [{lo3:+.4f}, {hi3:+.4f}]  p={fmt_p(p3)}")
            c = np.array([e["max_jaccard"] for e in rev + pool], float)
            qs = np.quantile(c, np.linspace(0, 1, NQ+1))
            c_low = float(c[c < qs[1]].mean())
            dl = dd[:, 1] + dd[:, 3]*(c_low - cm); est = b[1] + b[3]*(c_low - cm)
            llo, lhi, lp = ci_p(dl)
            entry["low_repetition"] = {"beta": float(est), "ci95": [llo, lhi], "p": lp}
            print(f"    low-repetition simple effect {est:+.4f}  "
                  f"[{llo:+.4f}, {lhi:+.4f}]  p={fmt_p(lp)}")
        full[tag] = entry
    results["full_pool"] = full

    # ---------------- READ, gated properly ---------------------------------
    print(f"\n{'#'*74}\n# READ\n{'#'*74}")
    d_rep = draw["rep"]; f_rep = full["rep"]["is_revision"]
    stable = d_rep["pct2_5"] > 0
    consistent = d_rep["frac_positive"] >= 0.95
    print(f"  across-draw 2.5th pct (repetition controlled): {d_rep['pct2_5']:+.4f}")
    print(f"  positive in {d_rep['frac_positive']:.0%} of {N_DRAWS} draws")
    print(f"  full-pool estimate: {f_rep['beta']:+.4f} "
          f"[{f_rep['ci95'][0]:+.4f}, {f_rep['ci95'][1]:+.4f}]")
    print()
    if not matched_ok:
        print("  MATCHING STILL FAILED. Whatever follows is confounded by window")
        print("  structure; the arms cannot be equated by construction either. Report")
        print("  the repetition-controlled estimate WITH the structural imbalance")
        print("  stated as a limitation, and do not lead with divergence.")
        print()
    if stable and consistent:
        print("  STABLE ACROSS DRAWS. The estimate stays positive in essentially every")
        print("  independent draw of the baseline, so it is not an artifact of one")
        print("  lucky set of random positions. Report the FULL-POOL estimate with")
        print("  cluster-robust CIs, and report the across-draw interval alongside it")
        print("  as the honest uncertainty.")
    elif consistent and not stable:
        print("  DIRECTIONALLY CONSISTENT BUT NOT STABLE. The sign holds in most draws")
        print("  but the across-draw interval includes zero, so the single-run CIs")
        print("  were too narrow. Report the ACROSS-DRAW interval and describe the")
        print("  effect as suggestive rather than established.")
    else:
        print("  NOT STABLE. The estimate changes sign or straddles zero across")
        print("  independent draws of the baseline. The divergence effect is NOT")
        print("  established: the earlier tight CIs reflected one draw, not the")
        print("  underlying uncertainty. Report the null.")
    print(f"\n  Note the earlier runs gave +0.0147 and +0.0093 from two draws of a")
    print(f"  revision-sized baseline. The spread above is why that happened, and")
    print(f"  why a count-matched baseline was the wrong design choice.")

    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    json.dump(results, open(OUT_PATH, "w"), indent=2, default=str)
    print(f"\nWrote {OUT_PATH}")


if __name__ == "__main__":
    main()
'''
ds = types.ModuleType("ds")
ds.DATA_ROOT = DATA_ROOT
exec(compile(DS_SOURCE, "<draw-stability functions>", "exec"), ds.__dict__)
print("arm builders loaded:", [n for n in ("build_revision", "build_pool", "extract_window",
                                            "max_jaccard", "pack", "eligible") if callable(getattr(ds, n, None))])

In [ ]:
# ---- eligibility with the repaired correctness flag (CORRECTNESS = "repaired") ----
# Replaces each loaded trace's `correct` with `correct_new` from notebook 07's
# repaired_ground_truth.json, so ds.eligible (correct and not truncated) uses it.
# Nothing else in the arm builders changes. A trace with no repaired verdict is
# counted and treated as not correct.
FLAG_STATS = Counter()
if CORRECTNESS == "repaired":
    _rep = json.load(open(EXPORT_DIR / "repaired_ground_truth.json"))
    _orig_load = ds.load_trace
    def _load_repaired(model, pid, cache):
        k = (model, pid)
        if k not in cache:
            tr = _orig_load(model, pid, {})
            if tr is not None:
                tr = dict(tr); r = _rep.get(f"{model}|{pid}")
                old = bool(tr.get("correct", False))
                if r is None:
                    FLAG_STATS["no repaired verdict (treated as not correct)"] += 1; tr["correct"] = False
                else:
                    tr["correct"] = bool(r["correct_new"])
                    FLAG_STATS["unchanged" if old == tr["correct"] else
                               ("now correct (was not)" if tr["correct"] else "now not correct (was)")] += 1
            cache[k] = tr
        return cache[k]
    ds.load_trace = _load_repaired
    print("eligibility: repaired correctness flag (correct_new)")
else:
    print("eligibility: saved `correct` flag (as reported)")


### Estimators

In [ ]:
def ls_fit(Y, X):
    b, *_ = np.linalg.lstsq(X, Y, rcond=None); return b

def cluster_boot(Y, X, groups, nb=N_BOOT, seed=SEED):
    beta = ls_fit(Y, X); rng = np.random.RandomState(seed)
    gi = defaultdict(list)
    for i, g in enumerate(groups): gi[g].append(i)
    arrs = [np.array(v) for v in gi.values()]; G = len(arrs)
    D = np.empty((nb, X.shape[1]))
    for k in range(nb):
        idx = np.concatenate([arrs[j] for j in rng.randint(0, G, G)])
        D[k] = ls_fit(Y[idx], X[idx])
    return beta, D, G

def ci_p(col):
    lo, hi = np.percentile(col, [2.5, 97.5])
    p = 2 * min((col <= 0).mean(), (col >= 0).mean())
    return float(lo), float(hi), float(min(1.0, max(p, 1.0 / len(col))))

def pfmt(p, nb=N_BOOT):
    """Never print a bootstrap floor as if it were measured."""
    return f"< {1.0/nb:.2g} (floor, {nb} resamples)" if p <= 1.0 / nb + 1e-12 \
           else f"{p:.4g}"

def boot_mean_diff(a, b, ga, gb, nb=N_BOOT, seed=SEED):
    rng = np.random.RandomState(seed)
    by = defaultdict(lambda: ([], []))
    for v, g in zip(a, ga): by[g][0].append(v)
    for v, g in zip(b, gb): by[g][1].append(v)
    keys = list(by); G = len(keys)
    out = []
    for _ in range(nb):
        pick = [keys[j] for j in rng.randint(0, G, G)]
        A = [v for k in pick for v in by[k][0]]
        B = [v for k in pick for v in by[k][1]]
        if A and B: out.append(np.mean(A) - np.mean(B))
    out = np.array(out)
    lo, hi = np.percentile(out, [2.5, 97.5])
    p = 2 * min((out <= 0).mean(), (out >= 0).mean())
    return float(lo), float(hi), float(min(1.0, max(p, 1.0 / nb))), int(G)

def auc(pos, neg):
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    allv = np.concatenate([pos, neg])
    order = np.argsort(allv)
    r = np.empty(len(allv), float); r[order] = np.arange(1, len(allv) + 1)
    sv = allv[order]; i = 0
    while i < len(sv):
        j = i
        while j + 1 < len(sv) and sv[j + 1] == sv[i]: j += 1
        if j > i: r[order[i:j + 1]] = r[order[i:j + 1]].mean()
        i = j + 1
    n1, n0 = len(pos), len(neg)
    return float((r[:n1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

## 1b — Build and embed the arms

In [ ]:
# ---- build all arms once and embed them, or reuse the saved per-event values ----
POOL_MULT = int(getattr(ds, "POOL_MULT", 4) or 4)

def slim(x, arm):
    return [{"arm": arm, "model": e["model"], "problem_id": str(e["problem_id"]),
             "trace_id": str(e["trace_id"]), "marker": e.get("marker", ""),
             "divergence": float(e["divergence"]), "rep": float(e["rep"])} for e in x]

if REUSE_CACHE and CACHE_PATH.exists():
    C = json.load(open(CACHE_PATH))
    rev, rnd4, rnd1 = C["revision"], C["random_pool"], C["random_count_matched"]
    rev_m, fil_m = C["revision_matched"], C["filler_matched"]
    print(f"reusing {CACHE_PATH.name}: " + ", ".join(f"{k}={len(v)}" for k, v in C.items()))
else:
    cache = {}
    rev, mlens = ds.build_revision(cache)
    print(f"\n  revision arm: {len(rev)} events, "
          f"{len({e['trace_id'] for e in rev})} traces")
    p_boundary = 1.0 - float(np.mean([e["frag_before"] for e in rev]))
    per_model = Counter(e["model"] for e in rev)
    big = Counter({k: POOL_MULT * v for k, v in per_model.items()})
    rnd4, _a = ds.build_pool(cache, random.Random(SEED), big, mlens, p_boundary)
    rnd1, _b = ds.build_pool(cache, random.Random(SEED), per_model, mlens, p_boundary)
    print(f"  random arm  : {len(rnd4)} events ({POOL_MULT}x pool, "
          f"p_boundary={p_boundary:.3f})")
    print(f"  robustness   : {len(rnd1)} events (count-matched single draw)")

    def norm_marker(t):
        t = ds.clean_bpe(str(t or "")).strip().lower()
        return re.split(r"[,\s]", t)[0] if t else ""

    fil, why = [], Counter()
    for model in getattr(ds, "MODELS", []):
        rp = Path(getattr(ds, "REJECT_DIR", REJECT_DIR)) / f"{model}_confirmed_filler.json"
        if not rp.exists(): why["no rejects file"] += 1; continue
        for f_ in json.load(open(rp)):
            pid = f_.get("problem_id")
            tr = ds.load_trace(model, pid, cache)
            if not ds.eligible(tr): why["trace not eligible"] += 1; continue
            fr = tr.get("full_response", "") or ""
            cs = f_.get("char_start"); matched = str(f_.get("matched_text") or "")
            if cs is None or not (0 <= cs < len(fr)):
                why["no usable char_start"] += 1; continue
            b, a = ds.extract_window(fr, cs, min(cs + len(matched), len(fr)))
            if b is None: why["window failed"] += 1; continue
            ev = ds.pack("filler", model, pid, b, a, cs, fr, tr)
            ev["marker"] = norm_marker(matched or f_.get("marker"))
            fil.append(ev); why["ok"] += 1
    print(f"  filler arm  : {len(fil)} events")
    for k, v in why.most_common():
        if k != "ok": print(f"      {v:>5}  {k}")

    lab_mk = {}
    for model in getattr(ds, "MODELS", []):
        md = Path(getattr(ds, "LABELS_DIR", LABELS_DIR)) / model
        if not md.exists(): continue
        for p in md.glob("*.json"):
            try: o = json.load(open(p))
            except Exception: continue
            lab_mk[(model, str(o.get("problem_id", p.stem)))] = [
                norm_marker(r.get("marker_phrase") or r.get("matched_text"))
                for r in (o.get("revisions") or []) if r.get("verified", True)]
    _ctr = Counter()
    for e in rev:
        k = (e["model"], str(e["problem_id"])); i = _ctr[k]; _ctr[k] += 1
        ms = lab_mk.get(k, []); e["marker"] = ms[i] if i < len(ms) else ""

    rng_m = np.random.RandomState(42)
    by_r, by_f = defaultdict(list), defaultdict(list)
    for e in rev: by_r[(e["model"], e["marker"])].append(e)
    for e in fil: by_f[(e["model"], e["marker"])].append(e)
    rev_m, fil_m = [], []
    for k in sorted(set(by_r) | set(by_f), key=str):
        A, B = by_r.get(k, []), by_f.get(k, [])
        n = min(len(A), len(B))
        if n == 0: continue
        rev_m += [A[i] for i in rng_m.choice(len(A), n, replace=False)]
        fil_m += [B[i] for i in rng_m.choice(len(B), n, replace=False)]
    print(f"  marker-matched: {len(rev_m)} per arm")

    from sentence_transformers import SentenceTransformer
    emb = SentenceTransformer(getattr(ds, "EMBED_MODEL_NAME", "all-MiniLM-L6-v2"))

    def score(events, label):
        done = 0
        for n, e in enumerate(events):
            if "divergence" in e: continue
            B = emb.encode(e["before"], normalize_embeddings=True)
            A = emb.encode(e["after"],  normalize_embeddings=True)
            cb, ca = B.mean(0), A.mean(0)
            e["divergence"] = 1.0 - float(np.dot(cb, ca) /
                              (np.linalg.norm(cb) * np.linalg.norm(ca) + 1e-12))
            e["rep"] = ds.max_jaccard(e["before"], e["after"])
            done += 1
            if done % 500 == 0:
                print(f"      {label}: {done}/{len(events)}", flush=True)
        return events

    print("\n  embedding")
    for lab, arm in (("revision", rev), (f"random{POOL_MULT}x", rnd4),
                     ("random1x", rnd1), ("revision-matched", rev_m),
                     ("filler-matched", fil_m)):
        score(arm, lab)
        print(f"    {lab:<18} n={len(arm):>6}  mean "
              f"{np.mean([e['divergence'] for e in arm]):.4f}")


    C = {"revision": slim(rev, "revision"), "random_pool": slim(rnd4, "random"),
         "random_count_matched": slim(rnd1, "random"),
         "revision_matched": slim(rev_m, "revision"), "filler_matched": slim(fil_m, "filler")}
    json.dump(C, open(CACHE_PATH, "w"))
    rev, rnd4, rnd1 = C["revision"], C["random_pool"], C["random_count_matched"]
    rev_m, fil_m = C["revision_matched"], C["filler_matched"]
    print(f"per-event values saved to {CACHE_PATH.name}")

## 2–3 — Three arms, robustness, decomposition

In [ ]:
def V(x): return np.array([e["divergence"] for e in x], float)
def T(x): return [e["trace_id"] for e in x]
def P(x): return [(e["model"], str(e["problem_id"])) for e in x]

hdr(f"THREE ARMS, ONE DESIGN ({POOL_MULT}x MIXTURE POOL)")
table = {}
for ckey, kf in (("trace", T), ("problem (model,pid)", P)):
    print(f"\n  clustered by {ckey}")
    print(f"  {'comparison':<34}{'n a/b':>13}{'diff':>10}{'95% CI':>22}"
          f"{'p':>28}")
    for lab, A, B in (("revision(m) vs filler(m)", rev_m, fil_m),
                      ("revision(m) vs random", rev_m, rnd4),
                      ("filler(m) vs random",    fil_m, rnd4),
                      ("revision(all) vs random", rev,  rnd4)):
        lo, hi, p, G = boot_mean_diff(V(A), V(B), kf(A), kf(B))
        d = float(np.mean(V(A)) - np.mean(V(B)))
        table.setdefault(ckey, {})[lab] = {
            "n_a": len(A), "n_b": len(B), "diff": d, "ci95": [lo, hi],
            "p": p, "clusters": G}
        print(f"  {lab:<34}{len(A)}/{len(B):<7}{d:>+10.4f}  "
              f"[{lo:+.4f}, {hi:+.4f}]{pfmt(p, N_BOOT):>28}")
    print(f"    clusters: {G}")

# robustness: the count-matched draw, same estimator
lo1, hi1, p1, _ = boot_mean_diff(V(rev), V(rnd1), T(rev), T(rnd1))
d1 = float(np.mean(V(rev)) - np.mean(V(rnd1)))
main = table["trace"]["revision(all) vs random"]
print(f"\n  ROBUSTNESS — count-matched single draw instead of the "
      f"{POOL_MULT}x pool")
print(f"    revision vs random   {d1:+.4f}  [{lo1:+.4f}, {hi1:+.4f}]  "
      f"p={pfmt(p1, N_BOOT)}")
print(f"    {POOL_MULT}x pool, same row  {main['diff']:+.4f}  "
      f"[{main['ci95'][0]:+.4f}, {main['ci95'][1]:+.4f}]  "
      f"p={pfmt(main['p'], N_BOOT)}")
rob = (main["ci95"][0] > 0) == (lo1 > 0)
print(f"    sign and significance agree across designs: {rob}")
if not rob:
    print("    -> revision vs random is NOT robust across baseline designs:")
    print("       the 4x pool and the count-matched draw disagree on significance.")

# decomposition
rf = table["trace"]["revision(m) vs filler(m)"]
rr = table["trace"]["revision(m) vs random"]
fr = table["trace"]["filler(m) vs random"]
print(f"\n  DECOMPOSITION (matched arms against the same pool)")
print(f"    revision - filler  {rf['diff']:+.4f}  "
      f"[{rf['ci95'][0]:+.4f}, {rf['ci95'][1]:+.4f}]")
print(f"      = (revision - random) {rr['diff']:+.4f}  "
      f"[{rr['ci95'][0]:+.4f}, {rr['ci95'][1]:+.4f}]")
print(f"      + (random - filler)   {-fr['diff']:+.4f}  "
      f"[{-fr['ci95'][1]:+.4f}, {-fr['ci95'][0]:+.4f}]")
print(f"      identity residual    {rf['diff']-(rr['diff']-fr['diff']):+.6f}")
if rf["diff"]:
    s = rr["diff"] / rf["diff"]
    print(f"    share from revisions ABOVE random: {s:6.1%}")
    print(f"    share from fillers BELOW random  : {1-s:6.1%}")

## 4 — Revision vs filler, repetition controlled

In [ ]:
# ---- revision vs filler with repetition as a covariate, three clusterings -------
# The paper reports the strictest: clusters = problem_id alone (a problem's
# events in all four models are resampled together).
ev = rev_m + fil_m
y  = np.array([e["divergence"] for e in ev], float)
c  = np.array([e["rep"] for e in ev], float)
x  = np.array([1.0 if str(e["arm"]).startswith("rev") else 0.0
               for e in ev], float)
cc = c - c.mean()
one = np.ones_like(y)
print(f"  {len(ev)} events, {int(x.sum())} revision / "
      f"{int(len(x)-x.sum())} filler")
print(f"  repetition: revision {c[x==1].mean():.4f}  "
      f"filler {c[x==0].mean():.4f}  "
      f"AUC {auc(c[x==0], c[x==1]):.4f}")

KEYS = [("trace_id", [str(e["trace_id"]) for e in ev]),
        ("(model, problem_id)",
         [f"{e['model']}|{e['problem_id']}" for e in ev]),
        ("problem_id ONLY", [str(e["problem_id"]) for e in ev])]
SPECS = [("divergence ~ arm",      [one, x]),
         ("  + repetition",        [one, x, cc]),
         ("  + repetition x arm",  [one, x, cc, x * cc])]
outA = {}
for kname, g in KEYS:
    print(f"\n  clustered by {kname} — {len(set(g))} clusters")
    print(f"  {'specification':<24}{'arm':>10}{'95% CI':>22}"
          f"{'p':>18}{'rep slope':>11}{'rep p':>10}")
    for lab, cols in SPECS:
        X = np.column_stack(cols)
        beta, D, G = cluster_boot(y, X, g)
        lo, hi, p = ci_p(D[:, 1])
        line = (f"  {lab:<24}{beta[1]:>+10.4f}  "
                f"[{lo:+.4f}, {hi:+.4f}]{pfmt(p):>18}")
        rb = rp = None
        if X.shape[1] > 2:
            rb = float(beta[2]); _, _, rp = ci_p(D[:, 2])
            line += f"{rb:>+11.4f}{pfmt(rp):>10}"
        print(line)
        outA.setdefault(kname, {})[lab.strip()] = {
            "arm": float(beta[1]), "ci95": [lo, hi], "p": p,
            "rep_slope": rb, "rep_p": rp, "clusters": G}

strict = outA["problem_id ONLY"]
base, ctrl, intr = (strict["divergence ~ arm"],
                    strict["+ repetition"],
                    strict["+ repetition x arm"])
print(f"\n  THE ANSWER, on the strictest clustering "
      f"({ctrl['clusters']} problems)")
print(f"    raw        {base['arm']:+.4f} "
      f"[{base['ci95'][0]:+.4f}, {base['ci95'][1]:+.4f}]  "
      f"p={pfmt(base['p'])}")
print(f"    + rep      {ctrl['arm']:+.4f} "
      f"[{ctrl['ci95'][0]:+.4f}, {ctrl['ci95'][1]:+.4f}]  "
      f"p={pfmt(ctrl['p'])}")
print(f"    + rep*arm  {intr['arm']:+.4f} "
      f"[{intr['ci95'][0]:+.4f}, {intr['ci95'][1]:+.4f}]  "
      f"p={pfmt(intr['p'])}")
c_alive = not (ctrl["ci95"][0] <= 0 <= ctrl["ci95"][1])
i_alive = not (intr["ci95"][0] <= 0 <= intr["ci95"][1])
if not c_alive:
    print("""
    -> With repetition controlled, the revision-filler difference is not
       distinguishable from zero on the strictest clustering.""")
elif c_alive and not i_alive:
    print("""
    -> The difference survives the repetition covariate but not the
       interaction: it is not stable once the arms may differ in how
       divergence falls with repetition.""")
else:
    print("""
    -> The difference survives both the covariate and the interaction on
       the strictest clustering. Read it together with the decomposition
       and the revision-vs-random row above.""")
R["repetition_covariate"] = outA

## 5 — Repetition by arm

In [ ]:
reps = {"revision (matched)": [e["rep"] for e in rev_m],
        "filler (matched)":   [e["rep"] for e in fil_m],
        "revision (all)":     [e["rep"] for e in rev],
        f"random {POOL_MULT}x": [e["rep"] for e in rnd4]}
print(f"\n  repetition by arm")
for k, v in reps.items():
    print(f"    {k:<20} n={len(v):>6}  mean {np.mean(v):.4f}  "
          f"median {np.median(v):.4f}")
a_rep = auc(reps["filler (matched)"], reps["revision (matched)"])
print(f"    repetition AUC, filler vs revision: {a_rep:.4f}")

In [ ]:
R["three_arm_table"] = table
R["pool_mult"] = POOL_MULT
R["robustness_count_matched"] = {"diff": d1, "ci95": [lo1, hi1], "p": p1}
R["repetition_by_arm"] = {k: {"n": len(v), "mean": float(np.mean(v))} for k, v in reps.items()}
R["repetition_auc_filler_vs_revision"] = a_rep
json.dump(R, open(OUT, "w"), indent=2, default=str)
print(f"\nWrote {OUT}")

In [ ]:
# ---- repaired flag vs saved flag, side by side ------------------------------
if CORRECTNESS == "repaired":
    print("traces loaded, by flag change:", dict(FLAG_STATS))
    _saved = EXPORT_DIR / "divergence_final.json"
    if not _saved.exists():
        raise SystemExit("divergence_final.json (saved flag) not found: run this notebook once with "
                         "CORRECTNESS = \"saved\" first, then again with \"repaired\".")
    S = json.load(open(_saved))
    hdr("SAVED FLAG (reported)  vs  REPAIRED FLAG")
    def row(lab, a, b):
        f = lambda d: (f"{d['diff']:+.4f} [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}] p={pfmt(d['p'])}  n {d['n_a']}/{d['n_b']}"
                       if d else "n/a")
        print(f"  {lab:<30} saved    {f(a)}\n  {'':<30} repaired {f(b)}")
    for ck in ("trace", "problem (model,pid)"):
        print(f"\n  three arms, clustered by {ck}")
        for lab in R["three_arm_table"][ck]:
            row(lab, S["three_arm_table"][ck].get(lab), R["three_arm_table"][ck][lab])
    so, ro = S["robustness_count_matched"], R["robustness_count_matched"]
    print(f"\n  robustness, count-matched draw: saved {so['diff']:+.4f} [{so['ci95'][0]:+.4f}, {so['ci95'][1]:+.4f}]"
          f"   repaired {ro['diff']:+.4f} [{ro['ci95'][0]:+.4f}, {ro['ci95'][1]:+.4f}]")
    print("\n  decomposition (trace clustering): revision-filler = (revision-random) + (random-filler)")
    for nm, T_ in (("saved", S["three_arm_table"]["trace"]), ("repaired", R["three_arm_table"]["trace"])):
        rf, rr, fr = (T_["revision(m) vs filler(m)"], T_["revision(m) vs random"], T_["filler(m) vs random"])
        sh = rr["diff"] / rf["diff"] if rf["diff"] else float("nan")
        print(f"    {nm:<9} {rf['diff']:+.4f} = {rr['diff']:+.4f} + {-fr['diff']:+.4f}   "
              f"share above random {sh:.1%}, below random {1-sh:.1%}")
    print("\n  repetition-controlled, clustered by problem_id only")
    for spec in ("divergence ~ arm", "+ repetition", "+ repetition x arm"):
        a = S["repetition_covariate"]["problem_id ONLY"][spec]; b = R["repetition_covariate"]["problem_id ONLY"][spec]
        print(f"    {spec:<22} saved {a['arm']:+.4f} [{a['ci95'][0]:+.4f}, {a['ci95'][1]:+.4f}] p={pfmt(a['p'])} ({a['clusters']} problems)"
              f"   repaired {b['arm']:+.4f} [{b['ci95'][0]:+.4f}, {b['ci95'][1]:+.4f}] p={pfmt(b['p'])} ({b['clusters']} problems)")
    R["flag_stats"] = dict(FLAG_STATS)
    json.dump(R, open(OUT, "w"), indent=2, default=str)
